In [ ]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns
from sodapy import Socrata
from datetime import datetime
import os
from tqdm import tqdm
import sys
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# sys.path.append("../src")
sys.path.append(str(Path.cwd().parent))
import utils

path = os.getcwd()
print(path)

path_repo = "C:/Users/PC/Desktop/tesis/"

c:\Users\PC\Desktop\python\rates_degree_work


In [ ]:
tasascap = pd.read_csv(path_repo + "/data/2022.csv", decimal = ",", thousands = ".", encoding='utf-8')
tasascap.columns = [x.lower().replace(" ","_") for x in tasascap.columns]
tasascap.rename(columns = {"tasa_efectiva_promedio_ponderada":"tasa", "montos_desembolsados":"monto", "numero_de_creditos_desembolsados":"num_obl", 
                           "sum_numero_de_creditos":"num_obl", "producto_de_crédito":"producto", "plazo_de_crédito":"plazo", "nombre_entidad":"banco"}, inplace=True)
tasascap.drop(columns = ["tipo_entidad", "codigo_entidad"], inplace =True)
tasascap.sample(5)

,nombre_tipo_entidad,banco,fecha_corte,tipo_de_persona,sexo,tamaño_de_empresa,tipo_de_crédito,tipo_de_garantía,producto,plazo,tasa,margen_adicional,monto,num_obl,grupo_etnico,antiguedad_de_la_empresa,tipo_de_tasa,rango_monto_desembolsado,clase_deudor,codigo_ciiu,codigo_municipio
74720,BC-ESTABLECIMIENTO BANCARIO,Banco Davivienda,04/11/2022,Jurídica,No aplica,Microempresa,Comercial preferencial o corporativo,Garantia idónea o no idónea,Construcción de proyectos de vivienda no vis (colocación en uvr),Más de 30 días y hasta 1 año,16.8300,5.0800,4242600000.0000,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42278,BC-ESTABLECIMIENTO BANCARIO,Itaú,09/09/2022,Jurídica,No aplica,Gran empresa,Comercial ordinario,Sin garantia,Empresarial en pesos,Más de 1 año y hasta 3 años,16.3500,0.0000,2500000000.0000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47233,BC-ESTABLECIMIENTO BANCARIO,Coopcentral,23/09/2022,Natural,Masculino,Microempresa,Microcredito,Sin garantia,Microcrédito entre 25 y 120 SMMLV,Más de 1 año y hasta 3 años,18.0000,0.0000,40000000.0000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85196,BC-ESTABLECIMIENTO BANCARIO,BBVA Colombia,25/11/2022,Natural,Femenino,No aplica,Vivienda,Garantia idónea o no idónea,Adquisición de vivienda vis (colocación en pesos),Más de 15 años y hasta 20 años,16.2300,0.0000,734434151.8200,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59163,BC-ESTABLECIMIENTO BANCARIO,Banco Caja Social S.A.,07/10/2022,Jurídica,No aplica,Mediana empresa,Comercial preferencial o corporativo,Sin garantia,Corporativo,Más de 30 días y hasta 1 año,12.5600,0.0000,1814166566.0000,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
cols_cat = ['nombre_tipo_entidad', 'banco', 'fecha_corte', 'tipo_de_persona', 'sexo', 'tamaño_de_empresa', 'tipo_de_crédito', 'tipo_de_garantía',
            'producto', 'plazo', 'grupo_etnico', 'antiguedad_de_la_empresa', 'tipo_de_tasa', 'clase_deudor', 'codigo_ciiu', 'codigo_municipio']
cols_cont = ['tasa', 'margen_adicional', 'monto', 'num_obl', 'rango_monto_desembolsado']

tasascap["fecha_corte"] = pd.to_datetime(tasascap["fecha_corte"], dayfirst=True, errors='coerce')

tasascap["tasa"] = tasascap["tasa"].astype(float)
tasascap["num_obl"] = tasascap["num_obl"].astype(int)
tasascap["fecha_corte"] = pd.to_datetime(tasascap["fecha_corte"])
tasascap["anomes"] = tasascap["fecha_corte"].dt.strftime("%Y-%m")

# tasascap = tasascap[~tasascap["banco"].isin(["Fondo Nacional del Ahorro", "La Hipotecaria", "Bancoldex", "Confiar",
#                                         "IRIS C.F.", "Bancamía S.A.", "Bancien", "Mibanco S.A.", "Banco Contactar",
#                                         'Tuya','Crezcamos','Credifamilia','Financiera Juriscoop C.F.','Rappipay',
#                                         'Banco GNB Sudameris','Banco Popular','Coopcentral','Banco Pichincha S.A.','Nu Colombia C.F.',
#                                         "Caja de Vivienda Militar y de Polícia"])] # Excluidos por tasas atipicos
                                        
# tasascap = tasascap[tasascap["tasa"] <= 50]
# tasascap = tasascap[tasascap["num_obl"] >= 2]
# tasascap = tasascap[(tasascap["rango_monto_desembolsado"].notna())] # tienen nulos antes de 202309

tasascap ["banco"] = tasascap["banco"].str.replace("Compañía De Financiamiento","CF").str.replace("Cooperativa Financiera","CF")
tasascap["plazo"] = tasascap["plazo"].replace({'Más de 1 año y hasta 3 años':"(1-3]", 'Más de 3 años y hasta 5 años':"(3-5]",
                                                'Más de 5 años y hasta 7 años':"(5-7]",
                                                'Más de 7 años y hasta 10 años':"(7-10]", 'Más de 10 años':"(10-]",
                                                "Hasta 6 meses":"(0-1]", "Más de 6 meses y hasta 1 año":"(0-1]",
                                                'Hasta 30 días':"(0-1]", 'Entre 31 días y 1 año':"(0-1]"})

tasascap["prom_monto_porobl"] = tasascap["monto"]/tasascap["num_obl"]

print(tasascap["fecha_corte"].min(), tasascap["fecha_corte"].max())

print(tasascap.shape)
print("Bancos:", tasascap["banco"].nunique())

tasascap.sample(7)

2022-07-01 00:00:00 2022-12-30 00:00:00
(107151, 23)
Bancos: 49


,nombre_tipo_entidad,banco,fecha_corte,tipo_de_persona,sexo,tamaño_de_empresa,tipo_de_crédito,tipo_de_garantía,producto,plazo,tasa,margen_adicional,monto,num_obl,grupo_etnico,antiguedad_de_la_empresa,tipo_de_tasa,rango_monto_desembolsado,clase_deudor,codigo_ciiu,codigo_municipio,anomes,prom_monto_porobl
87827,BC-ESTABLECIMIENTO BANCARIO,Bancolombia,2022-12-02,Natural,Masculino,No aplica,Consumo,Garantia idónea o no idónea,Consumo empleados,(1-3],12.8600,0.0000,4778900.0000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-12,4778900.0000
13037,BC-ESTABLECIMIENTO BANCARIO,Bancoomeva,2022-07-22,Natural,Femenino,No aplica,Consumo,Sin garantia,Consumo empleados,(5-7],31.9100,0.0000,300000.0000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07,300000.0000
4109,BC-ESTABLECIMIENTO BANCARIO,Banco Caja Social S.A.,2022-07-01,Jurídica,No aplica,Pequeña empresa,Comercial preferencial o corporativo,Sin garantia,Corporativo,(1-3],10.9900,0.0000,2948275828.0000,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07,327586203.1111
37103,BC-ESTABLECIMIENTO BANCARIO,Banco Caja Social S.A.,2022-09-02,Natural,Masculino,Pequeña empresa,Comercial ordinario,Garantía fondo nacional de garantías (FNG) o Fondo de Garantías de Antioquia (FGA),Otros Comercial,(1-3],21.2100,0.0000,190000000.0000,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-09,63333333.3333
67848,BC-ESTABLECIMIENTO BANCARIO,Bancolombia,2022-10-28,Natural,Masculino,No aplica,Consumo,Sin garantia,Libre inversión,(3-5],18.1100,0.0000,593561000.0000,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-10,45658538.4615
99129,BC-ESTABLECIMIENTO BANCARIO,Bancolombia,2022-12-23,Natural,Masculino,Mediana empresa,Comercial ordinario,Garantia idónea o no idónea,Factoring,(0-1],18.3700,0.0000,21000337.0000,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-12,10500168.5000
24882,BC-ESTABLECIMIENTO BANCARIO,Banagrario,2022-08-12,Natural,Femenino,Microempresa,Microcredito,Garantía fondo nacional de garantías (FNG) o Fondo de Garantías de Antioquia (FGA),Microcrédito menor a 25 SMMLV,(3-5],28.4100,0.0000,88200000.0000,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-08,29400000.0000


In [ ]:
utils.vc(tasascap["producto"])

AttributeError: module 'utils' has no attribute 'vc'

In [ ]:
tasascap.isnull().mean()

In [ ]:
print(np.percentile(tasascap["tasa"], 2.25))
print(np.percentile(tasascap["tasa"], 5))
print(np.percentile(tasascap["tasa"], 95))
print(np.percentile(tasascap["tasa"], 97.5))

In [ ]:
print(tasascap[tasascap["tasa"] <= 10].shape)
print(tasascap[tasascap["tasa"] >= 40].shape)

In [ ]:
# ax = tasascap[tasascap["fecha_corte"] >= "2025-01-01"]
ax = tasascap
ax = ax.groupby("producto").agg({"monto":"sum", "num_obl":"sum", "tasa":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_obl", ascending=False)#.head(30)
ax.columns = ["producto", "monto", "num_obl", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_obl"]/ax["num_obl"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_obl"]
ax = ax.sort_values("num_obl", ascending=False)#.head(30)
ax

In [ ]:
# ax = tasascap[tasascap["fecha_corte"] >= "2025-01-01"]
ax = tasascap
ax = ax.groupby("banco").agg({"monto":"sum", "num_obl":"sum", "tasa":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_obl", ascending=False)#.head(30)
ax.columns = ["banco", "monto", "num_obl", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_obl"]/ax["num_obl"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_obl"]
ax = ax.sort_values("num_obl", ascending=False)#.head(30)
ax

In [ ]:
tasascap["anomes"].value_counts(dropna = False, normalize =True)
ax = tasascap
ax = ax.groupby(["anomes"]).agg({"monto":"sum", "num_obl":"sum", "tasa":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_obl", ascending=False)#.head(30)
ax.columns = ["mes", "monto", "num_obl", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_obl"]/ax["num_obl"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_obl"]
#ax.sort_values("monto", ascending=False).head(40)
ax

In [ ]:
ax = tasascap
ax = ax.groupby(["anomes", "plazo", "rango_desembolso"]).agg({"monto":"sum", "num_obl":"sum", "tasa":"mean"}).reset_index()#.sort_values("num_obl", ascending=False)#.head(30)
ax["monto"] = ax["monto"]/1000000
ax["prom_monto_porobl"] = ax["monto"]/ax["num_obl"] # millones

#ax.sort_values("monto", ascending=False).head(40)
tasascap = ax
tasascap